# 14. RAG Agent 基礎版

使用 LangGraph 建立檢索增強生成 (RAG) Agent - 簡化版本。

---

## 🎯 學習目標

- ✅ 理解 RAG 的核心概念
- ✅ 使用關鍵字搜尋實現檢索
- ✅ 建立完整的 RAG 工作流程
- ✅ 無需 API Key 即可執行

---

## 📊 RAG 架構

```
┌─────────────────────────────────────────────────────────┐
│                    RAG 基礎架構                          │
├─────────────────────────────────────────────────────────┤
│                                                         │
│   ┌──────────┐     ┌──────────┐     ┌──────────┐       │
│   │ 用戶問題 │ ──▶ │  檢索器  │ ──▶ │  生成器  │       │
│   └──────────┘     └────┬─────┘     └────┬─────┘       │
│                         │                │              │
│                    ┌────▼────┐      ┌────▼────┐        │
│                    │ 知識庫  │      │  回答   │        │
│                    └─────────┘      └─────────┘        │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

### RAG = Retrieval + Augmented + Generation

| 步驟 | 說明 |
|------|------|
| **Retrieval** | 從知識庫檢索相關文件 |
| **Augmented** | 用檢索結果增強提示 |
| **Generation** | 根據增強提示生成回答 |

In [1]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
import re
from pathlib import Path

---

## 14.1 載入知識庫

我們使用一份 LangGraph 相關的知識文件作為知識庫：

In [2]:
# 載入知識庫文件
knowledge_path = Path("../data/langgraph_knowledge.md")

if knowledge_path.exists():
    with open(knowledge_path, "r", encoding="utf-8") as f:
        raw_content = f.read()
    print(f"✅ 知識庫載入成功: {len(raw_content)} 字元")
else:
    # 內建備用知識庫
    raw_content = """
# LangGraph 知識庫

## StateGraph
StateGraph 是 LangGraph 的核心類別，用於建構有狀態的圖。

## 節點 Node
節點是處理單元，接收 State 並返回更新。

## 邊 Edge
邊定義節點之間的連接關係。

## Reducer
Reducer 決定如何合併多個更新。add_messages 是常用的 Reducer。

## Checkpointer
Checkpointer 用於保存狀態，MemorySaver 是最簡單的實現。
"""
    print("⚠️ 使用內建備用知識庫")

✅ 知識庫載入成功: 4466 字元


---

## 14.2 文件切分

將大文件切分成小塊（chunks），方便檢索：

In [3]:
def split_into_chunks(text: str, chunk_size: int = 500) -> list[dict]:
    """將文本切分成小塊
    
    Args:
        text: 原始文本
        chunk_size: 每塊大小
    Returns:
        list[dict]: [{"id": 0, "content": "..."}, ...]
    """
    # 用兩個換行分隔段落
    paragraphs = re.split(r'\n\n+', text)
    
    chunks = []
    current_chunk = ""
    
    for para in paragraphs:
        if len(current_chunk) + len(para) < chunk_size:
            current_chunk += para + "\n\n"
        else:
            if current_chunk:
                chunks.append({"id": len(chunks), "content": current_chunk.strip()})
            current_chunk = para + "\n\n"
    
    if current_chunk:
        chunks.append({"id": len(chunks), "content": current_chunk.strip()})
    
    return chunks

# 切分知識庫
knowledge_chunks = split_into_chunks(raw_content)
print(f"✅ 切分完成: {len(knowledge_chunks)} 個區塊")
print(f"\n📄 範例區塊:")
print(f"   {knowledge_chunks[0]['content'][:100]}...")

✅ 切分完成: 10 個區塊

📄 範例區塊:
   # LangGraph 核心概念知識庫

## 什麼是 LangGraph？

LangGraph 是一個用於構建有狀態、多角色應用程式的框架，專為 LLM（大型語言模型）設計。它擴展了 LangCh...


---

## 14.3 關鍵字檢索器

使用 TF-IDF 風格的簡易檢索：

In [4]:
def keyword_search(query: str, chunks: list[dict], top_k: int = 3) -> list[dict]:
    """關鍵字搜尋
    
    計算每個區塊與查詢的關鍵字匹配分數
    """
    # 提取查詢關鍵字
    query_words = set(re.findall(r'\w+', query.lower()))
    
    results = []
    for chunk in chunks:
        content = chunk["content"].lower()
        
        # 計算匹配分數
        score = 0
        for word in query_words:
            # 完整匹配
            if word in content:
                score += content.count(word)
        
        if score > 0:
            results.append({
                "id": chunk["id"],
                "content": chunk["content"],
                "score": score
            })
    
    # 排序並返回 top_k
    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]

# 測試檢索
print("📊 測試檢索:")
test_results = keyword_search("StateGraph 如何使用", knowledge_chunks)
for r in test_results:
    print(f"  [分數: {r['score']}] {r['content'][:60]}...")

📊 測試檢索:
  [分數: 3] # LangGraph 核心概念知識庫

## 什麼是 LangGraph？

LangGraph 是一個用於構建有狀態...
  [分數: 1] graph = StateGraph(MyState)
graph.add_node("my_node", my_fun...
  [分數: 1] graph.add_conditional_edges("primary", check_fallback, {
   ...


---

## 14.4 定義 RAG 狀態

In [5]:
class RAGState(TypedDict):
    """RAG Agent 狀態"""
    question: str           # 用戶問題
    documents: list         # 檢索到的文件
    generation: str         # 生成的回答
    route: str              # 路由決策

print("✅ RAG 狀態定義完成")

✅ RAG 狀態定義完成


---

## 14.5 定義 RAG 節點

In [6]:
def route_question(state: RAGState) -> dict:
    """路由：決定是否需要檢索"""
    question = state["question"].lower()
    
    # 簡單規則：包含問句關鍵字就檢索
    retrieval_keywords = ["什麼", "如何", "怎麼", "為什麼", "是什麼", "what", "how", "why"]
    
    for kw in retrieval_keywords:
        if kw in question:
            print("  🔍 決定: 需要檢索")
            return {"route": "retrieve"}
    
    print("  💬 決定: 直接回答")
    return {"route": "direct"}

def retrieve(state: RAGState) -> dict:
    """檢索相關文件"""
    question = state["question"]
    print(f"  📚 檢索中: {question[:30]}...")
    
    docs = keyword_search(question, knowledge_chunks, top_k=3)
    print(f"  📄 找到 {len(docs)} 個相關區塊")
    
    return {"documents": docs}

def generate_answer(state: RAGState) -> dict:
    """根據文件生成回答（模擬 LLM）"""
    question = state["question"]
    docs = state.get("documents", [])
    
    print("  🤖 生成回答中...")
    
    if docs:
        # 模擬根據文件生成回答
        context = "\n".join([d["content"][:200] for d in docs])
        answer = f"根據知識庫，關於『{question}』的回答：\n\n"
        answer += f"📚 參考資料（{len(docs)} 個區塊）:\n"
        for d in docs:
            answer += f"  • {d['content'][:80]}...\n"
    else:
        answer = f"抱歉，知識庫中找不到與『{question}』相關的資訊。"
    
    return {"generation": answer}

def direct_answer(state: RAGState) -> dict:
    """直接回答（不檢索）"""
    print("  💡 直接回答")
    return {"generation": f"您好！您說：{state['question']}。有什麼我可以幫助的嗎？"}

print("✅ RAG 節點定義完成")

✅ RAG 節點定義完成


---

## 14.6 建構 RAG 圖

In [7]:
def route_decision(state: RAGState) -> Literal["retrieve", "direct"]:
    """路由決策"""
    return state.get("route", "retrieve")

# 建構圖
graph = StateGraph(RAGState)

# 添加節點
graph.add_node("router", route_question)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate_answer)
graph.add_node("direct", direct_answer)

# 添加邊
graph.add_edge(START, "router")
graph.add_conditional_edges("router", route_decision, {
    "retrieve": "retrieve",
    "direct": "direct"
})
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)
graph.add_edge("direct", END)

# 編譯
rag_app = graph.compile()
print("✅ RAG Agent 已就緒")

✅ RAG Agent 已就緒


In [8]:
print("📊 RAG 圖結構:")
print(rag_app.get_graph().draw_mermaid())

📊 RAG 圖結構:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	retrieve(retrieve)
	generate(generate)
	direct(direct)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	retrieve --> generate;
	router -.-> direct;
	router -.-> retrieve;
	direct --> __end__;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



---

## 14.7 測試 RAG Agent

In [9]:
print("🚀 測試 RAG Agent:")
print("=" * 60)

questions = [
    "StateGraph 是什麼？如何使用？",
    "什麼是 Reducer？",
    "你好！"
]

for q in questions:
    print(f"\n❓ 問題: {q}")
    print("-" * 40)
    
    result = rag_app.invoke({
        "question": q,
        "documents": [],
        "generation": "",
        "route": ""
    })
    
    print(f"\n💡 回答:\n{result['generation']}")
    print("\n" + "=" * 60)

🚀 測試 RAG Agent:

❓ 問題: StateGraph 是什麼？如何使用？
----------------------------------------
  🔍 決定: 需要檢索
  📚 檢索中: StateGraph 是什麼？如何使用？...
  📄 找到 3 個相關區塊
  🤖 生成回答中...

💡 回答:
根據知識庫，關於『StateGraph 是什麼？如何使用？』的回答：

📚 參考資料（3 個區塊）:
  • # LangGraph 核心概念知識庫

## 什麼是 LangGraph？

LangGraph 是一個用於構建有狀態、多角色應用程式的框架，專為 LLM（大...
  • graph = StateGraph(MyState)
graph.add_node("my_node", my_function)
graph.add_edg...
  • graph.add_conditional_edges("primary", check_fallback, {
    "done": END,
    "f...



❓ 問題: 什麼是 Reducer？
----------------------------------------
  🔍 決定: 需要檢索
  📚 檢索中: 什麼是 Reducer？...
  📄 找到 3 個相關區塊
  🤖 生成回答中...

💡 回答:
根據知識庫，關於『什麼是 Reducer？』的回答：

📚 參考資料（3 個區塊）:
  • graph = StateGraph(MyState)
graph.add_node("my_node", my_function)
graph.add_edg...
  • ```python
from langgraph.graph.message import add_messages

class State(TypedDic...
  • # LangGraph 核心概念知識庫

## 什麼是 LangGraph？

LangGraph 是一個用於構建有狀態、多角色應用程式的框架，專為 LLM（大...



❓ 問題: 你好！
----------------------------------------
  💬 決定: 直接回答
  💡 直接回答


---

## 💡 重點回顧

### RAG 工作流程

```
1. 用戶問題 → 路由判斷
2. 需要檢索 → 關鍵字搜尋 → 取得相關文件
3. 根據文件 → 生成回答
```

### 關鍵組件

| 組件 | 職責 |
|------|------|
| Router | 決定是否需要檢索 |
| Retriever | 從知識庫搜尋相關文件 |
| Generator | 根據文件生成回答 |

---

下一步：[15. RAG Agent 進階版](15_rag_agent_advanced.ipynb)